# Aula 2 — Modelo de posição, atrito e o primeiro projeto

**Projeto final · Laboratório de Controle Automático · Ifes Guarapari · 18/11 · 1h40**

**Objetivo da aula:** passar do modelo de **velocidade** (Aula 1) para o de **posição**, medir as
não linearidades da bancada (zona morta e atrito estático), montar o **gêmeo digital** e fazer o
primeiro projeto de controlador pelo **lugar das raízes** — e ver o que acontece com ele no motor
de verdade.

**Ao final desta aula a bancada terá:** o modelo de posição $G_p(s)$ e as não linearidades salvos ·
o gêmeo digital validado · o projeto da iteração 1 testado na bancada. **Fecha o Trabalho 1.**

**Papéis na bancada (troquem a cada aula):** 🔧 **operador(a)** — mexe no hardware e grava o
firmware · 💻 **programador(a)** — roda e edita o notebook · ✍️ **relator(a)** — escreve as
respostas ❓ e confere os números. Com 2 pessoas, o relator acumula com o programador.

**Legenda:** ⏱️ tempo sugerido · 📘 teoria (leia antes de fazer) · 🔧 bancada · 💻 notebook ·
✏️ complete o código · ❓ responda · ✅ checkpoint — **chame o professor**.

| ⏱️ | Parte | O que se faz |
|---|---|---|
| 0–30 min | 1 | Ensaios de malha aberta e de partida |
| 30–45 min | 2 | Modelo de posição e gêmeo digital |
| 45–90 min | 3 | Projeto pelo lugar das raízes — iteração 1 |
| 90–100 min | ✅ | Checkpoint 2 (fim do Trabalho 1) |

In [ ]:
# ⚙️ CONFIGURAÇÃO — rode esta célula sempre que abrir o notebook
import numpy as np
import matplotlib.pyplot as plt
import control as ct
from scipy.optimize import minimize
from ferramentas_projeto import *

PORTA = "/dev/ttyUSB0"   # ✏️ porta do Arduino (veja a lista impressa abaixo; Uno original: "/dev/ttyACM0")
BANCADA = "1"            # ✏️ número da bancada

s = ct.tf("s")
Ts = 0.02          # período de amostragem do firmware [s]
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})
P = carregar_parametros()          # resultados das aulas anteriores (dados/parametros_bancada.json)
print("bancada", BANCADA, "| parâmetros já salvos:", list(P))
print("Arduinos conectados:"); portas()

## Parte 1 — Ensaios de malha aberta e de partida · ⏱️ 30 min

📘 **Da velocidade à posição.** A posição é a **integral** da velocidade. No domínio de Laplace,
integrar é dividir por $s$; então o modelo de posição é o de velocidade com um integrador:
$$G_p(s) = \frac{\Theta(s)}{U(s)} = \frac{K_m}{s\,(\tau s + 1)}\;e^{-Ls}\qquad\left[\frac{\text{graus}}{\text{PWM}}\right]$$
$K_m$ é o ganho de velocidade em **graus/s por PWM**. Com um degrau de PWM de amplitude $A$, depois
do transitório a velocidade fica constante e o ângulo cresce como uma **reta**, de inclinação
$$a(A) = K_m\,(A - F_c)\qquad (A > 0),$$
onde $F_c$ é a **zona morta** (o PWM que o atrito "come" com o motor em movimento). Com **duas**
amplitudes, a diferença entre as inclinações elimina $F_c$:
$$K_m = \frac{a(A_2) - a(A_1)}{A_2 - A_1},\qquad F_c = A_1 - \frac{a(A_1)}{K_m}.$$
Fazemos isso nos dois sentidos de giro, porque o atrito pode não ser simétrico.

**🔧💻 Passo 1.1** — quatro degraus de PWM: +100, +200, −100, −200 (≈ 4 s cada; a roda dá algumas
voltas). Roda livre!

---
🎓 **Explicação para a turma — da velocidade para a posição — e por que duas amplitudes**

O ângulo é a **integral** da velocidade. O caminho do PWM até o ângulo tem quatro blocos:

1. **zona morta** — o atrito “come” $F_c$ do PWM (não linear: fica fora de $G_p$);
2. **velocidade** — o FOPDT da Aula 1, em contas por janela de 20 ms;
3. **conversão** — dividir por $T_s\,\mathrm{CPG}$ passa de contas/20 ms para graus/s;
4. **integrador** $1/s$ — de graus/s para graus.

Os três blocos lineares formam $G_p(s) = K_m e^{-Ls}/[s(\tau s+1)]$, com $K_m = K/(T_s\,\mathrm{CPG})$.

![Figura 1 — Do PWM ao ângulo: a zona morta, o modelo de velocidade da Aula 1, a conversão de unidade e o integrador; embaixo, o que cada sinal faz após um degrau de PWM.](figuras/a2_velocidade_para_posicao.png)

*Figura 1 — O modelo de posição é o de velocidade seguido de um integrador. Embaixo, cada sinal após um degrau de PWM: a velocidade sobe em 1ª ordem e o ângulo vira uma reta.*

A Aula 1 já **prevê** $K_m$ (≈ 5,96 na bancada virtual), mas o ensaio de posição o **mede**
diretamente (5,66): usamos o medido, porque o $K$ da Aula 1 foi ajustado só na faixa 60–220 do PRBS.

Em malha aberta, depois do transitório, o ângulo vira uma **reta**: a inclinação $a$ é a
velocidade de regime. Com o atrito, $a = K_m\,(A - F_c)$ — e uma reta tem **dois** parâmetros.
Com uma amplitude só não dá para separar o ganho $K_m$ da zona morta $F_c$; com **duas**
amplitudes no mesmo sentido, sim:
$$K_m = \frac{a_{200} - a_{100}}{200 - 100},\qquad F_c = 100 - \frac{a_{100}}{K_m}.$$

![Figura 2 — Os quatro ensaios de malha aberta e a reta a(A) de cada sentido (dados da bancada virtual).](figuras/a2_duas_amplitudes.png)

*Figura 2 — Os quatro ensaios de malha aberta e a reta a(A) de cada sentido (dados da bancada virtual).*

O Trabalho 3 usava $k' = a/(K\cdot A)$ com uma amplitude só; isso **embute** a zona morta no
ganho e o subestima (aqui daria $K_m$ ≈ 3,8 em vez de 5,7, ≈ 32 % baixo). O método de duas
amplitudes corrige isso.

---

In [ ]:
ens = {}
for nome, A in [("ma_p100", 100), ("ma_p200", 200), ("ma_m100", -100), ("ma_m200", -200)]:
    ens[A] = coletar(nome, PORTA, MODO=1, PWM_MA=A, CPG=P["CPG"])
plt.figure(figsize=(8, 3.5))
for A, d in ens.items():
    plt.plot(d["t"], d["angle"], label=f"PWM = {A:+d}")
plt.axvspan(1.5, 2.5, color="0.9", zorder=0, label="trecho de regime (reta)")
plt.xlabel("tempo (s)"); plt.ylabel("ângulo (graus)"); plt.legend(fontsize=8); plt.show()

📘 **Atrito estático.** Parado, o motor precisa de **mais** PWM para começar a girar ($F_s$) do que
para continuar girando ($F_c$) — o atrito "gruda". O MODO 4 aumenta o PWM devagar (10 PWM por
segundo) e para assim que a roda se mexe: o PWM naquele instante é $F_s$.

**🔧💻 Passo 1.2** — teste de partida nos dois sentidos (até ≈ 8 s cada).

---
🎓 **Explicação para a turma — os dois atritos: em movimento e parado**

- **Coulomb ($F_c$):** com a roda girando, uma parte fixa do PWM é “gasta” com o atrito —
  é a zona morta da reta $a(A)$.
- **Estático ($F_s$):** parada, a roda precisa de **mais** PWM para começar a girar. O teste de
  partida (MODO 4) sobe o PWM em rampa lenta e registra o valor em que o ângulo saiu do lugar.

![Figura 3 — Modelo de atrito usado no gêmeo (esq.) e o teste de partida (dir.).](figuras/a2_atrito.png)

*Figura 3 — Modelo de atrito usado no gêmeo (esq.) e o teste de partida (dir.).*

Por que isso vai importar: quando a roda **para** perto da referência, o PID precisa produzir mais
de $F_s$ para movê-la de novo. Com erro pequeno e $k_p$ pequeno, isso não acontece — a roda
“cola” (Parte 3).

---

In [ ]:
Fs = {}
for sinal, nome in [(+1, "partida_pos"), (-1, "partida_neg")]:
    d = coletar(nome, PORTA, MODO=4, SINAL=sinal, RAMPA=10, CPG=P["CPG"])
    i = np.argmax(np.abs(d["angle"]) > 0)          # primeira amostra com movimento
    Fs[sinal] = abs(d["pwm"][i])
    print(f"sentido {sinal:+d}: a roda saiu do repouso com PWM ≈ {Fs[sinal]:.0f}")

## Parte 2 — Modelo de posição e gêmeo digital · ⏱️ 15 min

**💻 Passo 2.1** — inclinações, $K_m$ e zona morta. Complete as duas linhas ✏️ com as fórmulas da
caixa 📘 da Parte 1 (o sentido negativo já está pronto, como modelo).

In [ ]:
a = {A: inclinacao(d, 1.5, 2.5) for A, d in ens.items()}      # graus/s
for A in a:
    print(f"A = {A:+4d} PWM  ->  inclinação a = {a[A]:7.1f} graus/s")

Km_pos = ...                         # ✏️ (a(200) − a(100)) / (200 − 100)
Fc_pos = ...                         # ✏️ 100 − a(100)/Km_pos
Km_neg = (a[-200] - a[-100]) / (-200 - (-100))
Fc_neg = -(-100 - a[-100] / Km_neg)
Km = (Km_pos + Km_neg) / 2
print(f"\nsentido +: Km = {Km_pos:.3f} graus/s/PWM, zona morta Fc = {Fc_pos:.1f} PWM, partida Fs = {Fs[+1]:.0f}")
print(f"sentido −: Km = {Km_neg:.3f} graus/s/PWM, zona morta Fc = {Fc_neg:.1f} PWM, partida Fs = {Fs[-1]:.0f}")

# ligação com a Aula 1: Km = K · k'
k_linha = Km / P["K"]
k_linha_teo = 1 / (Ts * P["CPG"])    # (contas/janela) -> graus/s
print(f"\nk' = Km/K = {k_linha:.1f}   ×   k' teórico = 1/(Ts·CPG) = {k_linha_teo:.1f}")

❓ **2.1** (a) A zona morta é igual nos dois sentidos? (b) Compare $F_s$ com $F_c$: o que isso diz
sobre o atrito? (c) Por que $k'$ teórico vale $1/(T_s\cdot\text{CPG})$?

✍️ **Resposta da bancada:** *(escreva aqui)*

📘 **Gêmeo digital.** É um programa que imita a bancada: o modelo $G_p(s)$ **mais** as não
linearidades medidas (zona morta, atrito estático, saturação em ±255, encoder com resolução
1/CPG) **mais** exatamente as mesmas contas que o firmware faz a cada 20 ms. A função `simular()`
recebe as mesmas chaves de `coletar()`: dá para testar qualquer ensaio no computador antes de
rodá-lo na bancada.

**💻 Passo 2.2** — monte o gêmeo e confira: ele tem que reproduzir os ensaios de malha aberta.

---
🎓 **Explicação para a turma — o gêmeo digital por dentro**

O gêmeo junta tudo o que medimos num simulador que roda **o mesmo algoritmo do firmware** a cada
20 ms, alimentando um motor simulado com passo fino (0,5 ms):

![Figura 4 — O que o gêmeo digital simula, bloco por bloco.](figuras/a2_gemeo_digital.png)

*Figura 4 — O que o gêmeo digital simula, bloco por bloco.*

- O modelo **linear** (só $G_p$) prevê o comportamento “ideal”. O gêmeo acrescenta atrito
  (com colagem), quantização do encoder, saturação e anti-windup — as não linearidades.
- Validamos o gêmeo comparando com um ensaio que **não** foi usado para ajustá-lo (gráfico desta
  parte). Se ele reproduz a bancada, podemos confiar nas previsões do projeto.
- Quando o gêmeo e a bancada discordarem, a bancada tem razão: o gêmeo é só tão bom quanto as
  medidas que o alimentam.

---

In [ ]:
GEMEO = dict(Km=Km, tau=P["tau"], L=max(P["L"] - Ts / 2, 0.0), Fc=Fc_pos, Fc_neg=Fc_neg,
             Fs=Fs[+1], Fs_neg=Fs[-1], cpg=P["CPG"])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for i, A in enumerate([100, -200]):
    sim = simular(GEMEO, MODO=1, PWM_MA=A)
    ax[i].plot(ens[A]["t"], ens[A]["angle"], color="0.6", lw=3, label="bancada")
    ax[i].plot(sim["t"], sim["angle"], "b", lw=1.3, label="gêmeo digital")
    ax[i].set_title(f"PWM = {A:+d}"); ax[i].set_xlabel("tempo (s)"); ax[i].legend()
ax[0].set_ylabel("ângulo (graus)"); plt.show()
P = salvar_parametros(Km=Km, Fc_pos=Fc_pos, Fc_neg=Fc_neg, Fs_pos=Fs[+1], Fs_neg=Fs[-1], GEMEO=GEMEO)

> Por que `L = P["L"] − Ts/2` no gêmeo? Metade do atraso medido na Aula 1 veio da **janela** de
> medida da velocidade, que não existe quando medimos posição (o ângulo é lido no instante da
> amostra). O gêmeo já simula a amostragem de 20 ms por conta própria.

## Parte 3 — Projeto pelo lugar das raízes: iteração 1 · ⏱️ 45 min

Vamos projetar um controlador que atenda às especificações 2 e 3 **e nada mais** — é o primeiro
projeto que qualquer engenheiro faria. Depois vamos testá-lo no gêmeo e **na bancada**.

📘 **Das especificações ao plano-s.** Um sistema de 2ª ordem com polos $s = -\sigma \pm j\omega_d$
tem
$$\zeta = \frac{-\ln(M_p)}{\sqrt{\pi^2 + \ln^2(M_p)}}\quad (M_p \text{ em fração}),\qquad
t_s\,(5\,\%) \approx \frac{3}{\sigma},\qquad \omega_d = \sigma\,\tan(\arccos\zeta).$$
Escolher $M_p$ e $t_s$ é escolher onde queremos os polos de malha fechada: o **polo desejado**
$s_d = -\sigma + j\omega_d$.

📘 **Lugar das raízes (LGR).** Com um ganho $k$ na frente da planta, os polos de malha fechada são
as raízes de $1 + k\,G(s) = 0$. Quando $k$ vai de 0 a ∞, essas raízes desenham curvas no plano-s —
o lugar das raízes. Um ponto $s$ está no LGR se, e somente se,
$$\angle G(s) = -180°\quad\text{(condição de ângulo)},\qquad\text{e o ganho é}\quad k = \frac{1}{|G(s)|}\quad\text{(condição de módulo)}.$$
O ângulo de $G$ num ponto é a soma dos ângulos dos vetores **zero → ponto** menos a soma dos
ângulos dos vetores **polo → ponto**.

📘 **Com atraso.** $e^{-Ls}$ em $s_d = -\sigma + j\omega_d$ tem módulo $e^{\sigma L}$ e ângulo
$-\omega_d L$ (em radianos). O atraso só **tira** ângulo. No laço digital usamos
$L_{ef} = L_{gêmeo} + T_s/2$: o segurador do PWM (ZOH) equivale a meio período de atraso.

📘 **A decisão:** se $\angle G(s_d) > -180°$, o LGR do ganho puro já passa à esquerda de $s_d$ —
não é preciso avanço de fase (PD); basta o ganho. Se $\angle G(s_d) < -180°$, falta ângulo e é
preciso um PD (Aula 3).

**💻 Passo 3.1** — metas da iteração 1: $M_p$ = 10 % e $t_s$ = 1,0 s.

---
🎓 **Explicação para a turma — especificações no plano-s e a condição de ângulo**

As metas ($M_p$, $t_s$) viram um **lugar** no plano-s: $M_p$ fixa o ângulo da reta que sai da
origem (ζ) e $t_s$ fixa a distância ao eixo imaginário (σ).

![Figura 5 — Região das especificações e os polos desejados das duas iterações.](figuras/a2_plano_s_especificacoes.png)

*Figura 5 — Região das especificações e os polos desejados das duas iterações.*

O LGR passa por $s_d$ se a soma dos ângulos for −180°. Para a iteração 1:

![Figura 6 — Os vetores que entram na conta do Passo 3.2, com os números do gabarito.](figuras/a2_condicao_angulo.png)

*Figura 6 — Os vetores que entram na conta do Passo 3.2, com os números do gabarito.*

**Leitura da conta:** cada polo “rouba” o ângulo do seu vetor até $s_d$; o atraso rouba
$\omega_d L_{ef}$ (em radianos). Se o total ainda é **maior** que −180°, o LGR do ganho puro
passa à esquerda de $s_d$ e só o ganho resolve; se for **menor**, falta ângulo e é preciso um
zero (PD) — que é o caso da iteração 2, na Aula 3.

---

In [ ]:
Mp_alvo = 10          # %
ts_alvo = 1.0         # s
zeta = ...            # ✏️ fórmula da caixa 📘 (cuidado: Mp em fração!)
sigma = ...           # ✏️ 3 / ts
wd = sigma * np.tan(np.arccos(zeta))
sd = complex(-sigma, wd)
print(f"zeta = {zeta:.3f}   sigma = {sigma:.2f}   wd = {wd:.2f}   ->   s_d = {sd.real:.2f} + j{sd.imag:.2f}")

**💻 Passo 3.2** — o ângulo de $G_p$ em $s_d$, parcela por parcela. $G_p$ tem um polo em $s = 0$
(o integrador) e um em $s = -1/\tau$ (o polo mecânico), mais o atraso.

In [ ]:
Km, tau = P["Km"], P["tau"]
L_ef = GEMEO["L"] + Ts / 2
G = Km / (s * (tau * s + 1))
ang_integrador = -np.degrees(np.angle(sd - 0))
ang_polo_mec = -np.degrees(np.angle(sd + 1 / tau))
ang_atraso = -np.degrees(wd * L_ef)
ang_total = ang_integrador + ang_polo_mec + ang_atraso
avanco = -180 - ang_total
print(f"polo em 0       : {ang_integrador:7.1f}°")
print(f"polo em −1/tau  : {ang_polo_mec:7.1f}°   (−1/tau = {-1/tau:.2f})")
print(f"atraso L_ef     : {ang_atraso:7.1f}°   (L_ef = {1e3*L_ef:.1f} ms)")
print(f"∠G(s_d) total   : {ang_total:7.1f}°   ->   ângulo que falta (avanço) = {avanco:.1f}°")

❓ **3.1** É preciso avanço de fase (PD) para estas metas? Justifique com o número acima.

✍️ **Resposta da bancada:** *(escreva aqui)*

📘 **PI.** Uma planta com integrador já segue degraus sem erro **no modelo linear**. Mas o atrito
real "prende" o eixo antes da referência; um integrador no controlador acumula esse erro e
empurra até vencer o atrito. O controlador PI é
$$C(s) = k\,\frac{s + z}{s} = k + \frac{k\,z}{s}\quad\Rightarrow\quad k_p = k,\ k_i = k\,z.$$
Pondo o zero $z$ **bem perto da origem** (regra prática: $z = \sigma/10$), o par polo-zero do PI
quase não muda o ângulo em $s_d$ — o LGR perto de $s_d$ fica como estava. O ganho sai da condição
de módulo, agora com o PI incluído: $k = 1\,/\,\big|G(s_d)\,e^{-L_{ef}s_d}\,(s_d+z)/s_d\big|$.

**💻 Passo 3.3** — ganhos da iteração 1.

In [ ]:
z = sigma / 10
Gsd = complex(ct.evalfr(G, sd)) * np.exp(-L_ef * sd)      # G(s_d) com o atraso
k = 1 / abs(Gsd * (sd + z) / sd)                          # condição de módulo
kp1 = ...             # ✏️
ki1 = ...             # ✏️
kd1 = 0.0
print(f"zero do PI em −{z:.2f};  k = {k:.3f}  ->  kp = {kp1:.3f} PWM/grau, ki = {ki1:.3f} PWM/(grau·s)")

---
🎓 **Explicação para a turma — o LGR da iteração 1**

![Figura 7 — LGR da iteração 1 (com Padé de L_ef) e os polos de malha fechada com o k calculado.](figuras/a2_lgr_iteracao1.png)

*Figura 7 — LGR da iteração 1 (com Padé de L_ef) e os polos de malha fechada com o k calculado.*

- O par polo (0) – zero (−0,3) do PI fica tão perto da origem que, visto de $s_d$, os dois vetores
  têm quase o mesmo ângulo: o PI **não estraga** o LGR perto de $s_d$ (zoom à direita).
- Como sobrava ângulo (−158,8°), os polos dominantes com esse $k$ caem um pouco à **esquerda** de
  $s_d$. O terceiro polo, perto do zero do PI, é **lento**: ele explica o $t_s$ de 1,9 s do modelo
  linear (maior que a meta de 1 s) — a “cauda” que o integrador leva para fechar o último pedaço.
- O LGR é desenhado para o **modelo linear**. Ele não sabe nada de atrito — e é aí que a
  iteração 1 vai falhar.

---

📘 **Margens (só o essencial).** `analisa_pid()` informa a **margem de fase** (PM) e a de **ganho**
(GM) da malha: quanto de atraso extra (PM, em graus) ou de ganho extra (GM, em dB) a malha
aguentaria antes de oscilar sem parar. Regra prática: PM ≈ 100·ζ; **PM entre 45° e 60°** é uma
boa faixa. Leitura de apoio: teoria M3 §3.1.8–3.1.9.

**💻 Passo 3.4** — a iteração 1 no modelo linear e no gêmeo digital (perfil completo, 17 s).

In [ ]:
it1 = analisa_pid(G, L_ef, kp1, ki1, kd1, gemeo=GEMEO, nome="Iteração 1")
lin = simular(dict(Km=Km, tau=tau, L=GEMEO["L"]), MODO=3, KP=kp1, KI=ki1, KD=kd1)
plota_ensaio(lin, it1["sim"], rotulos=["modelo linear (sem atrito)", "gêmeo digital (com atrito)"],
             titulo="Iteração 1", mostra_u_pid=False)
plt.show()

**🔧💻 Passo 3.5** — e na bancada? O perfil completo dura 17 s.

In [ ]:
CFG1 = dict(MODO=3, CPG=P["CPG"], KP=kp1, KI=ki1, KD=kd1, N=30)
mf1 = coletar("mf_iter1", PORTA, **CFG1)
imprime_metricas(metricas_ensaio(mf1), "BANCADA — iteração 1")
plota_ensaio(it1["sim"], mf1, rotulos=["gêmeo", "bancada"], titulo="Iteração 1: gêmeo × bancada")
plt.show()
print(f"\nFaixa de 'colagem': Fs/kp = {P['Fs_pos']:.0f}/{kp1:.2f} = {P['Fs_pos']/kp1:.0f} graus")
P = salvar_parametros(iter1=dict(kp=kp1, ki=ki1, kd=kd1))

❓ **3.2** A iteração 1 atende às especificações no modelo linear? E no gêmeo e na bancada? O que
aconteceu com o eixo perto da referência?

✍️ **Resposta da bancada:** *(escreva aqui)*

❓ **3.3** Explique com a "faixa de colagem" $F_s/k_p$ impressa acima. O que seria preciso mudar
no projeto para o atrito prender menos?

✍️ **Resposta da bancada:** *(escreva aqui)*

---
🎓 **Explicação para a turma — a faixa de colagem**

Com a roda **parada**, ela só volta a andar se $|u| > F_s$. Enquanto o integrador não cresce, quem
produz o PWM é o termo proporcional $k_p\,e$. Logo existe uma faixa de erro
$$|e| < \frac{F_s}{k_p}$$
em que a roda fica **parada fora da referência** até o integrador (lento) acumular o que falta.

![Figura 8 — Faixa de colagem da iteração 1 (≈ 68°) e do controlador final (≈ 18°).](figuras/a2_faixa_colagem.png)

*Figura 8 — Faixa de colagem da iteração 1 (≈ 68°) e do controlador final (≈ 18°).*

Com $k_p$ = 0,71 a faixa é enorme: a roda cola dezenas de graus antes da referência, e o
integrador com $k_i$ = 0,21 leva segundos para tirá-la dali — o que se vê no gêmeo e na bancada.
O remédio (Aula 3): polos mais rápidos (⇒ $k_p$ maior) **e** compensar o atrito diretamente
(`ZM`).

---

## ✅ Checkpoint 2 — fim do Trabalho 1 · ⏱️ 10 min

Rode a célula abaixo, salve o notebook e chame o professor.

In [ ]:
P = carregar_parametros()
print(f"Bancada {BANCADA} — Trabalho 1 (Aulas 1 e 2)")
print(f"  CPG = {P['CPG']:.4f} contas/grau")
print(f"  velocidade: K = {P['K']:.4f}, tau = {P['tau']:.3f} s, L = {1e3*P['L']:.1f} ms, u0 = {P['u0']:.1f}")
print(f"  posição: Km = {P['Km']:.3f} graus/s/PWM")
print(f"  zona morta Fc = +{P['Fc_pos']:.1f} / −{P['Fc_neg']:.1f} PWM;  partida Fs = +{P['Fs_pos']:.0f} / −{P['Fs_neg']:.0f} PWM")
print(f"  iteração 1: kp = {P['iter1']['kp']:.3f}, ki = {P['iter1']['ki']:.3f}")

---
🎓 **Explicação para a turma — fechamento do Trabalho 1**

O Trabalho 1 termina aqui. Ele deve mostrar, **com os dados da sua bancada**:

1. o modelo de velocidade (Aula 1) e o de posição com $K_m$, $F_c^\pm$, $F_s^\pm$;
2. a validação do gêmeo digital (gráfico gêmeo × bancada);
3. o projeto da iteração 1 (conta de ângulo, ganhos, LGR, margens);
4. o confronto linear × gêmeo × bancada e a explicação pela faixa de colagem.

A mensagem principal: **um projeto certo para o modelo linear pode falhar na planta real** — e o
modelo das não linearidades (gêmeo) já era capaz de prever isso antes de ligar o motor.

---